# ERA5 Land Extraction

In [1]:
'''
06_00_ERA5_extraction.ipynb
ERA5 data extraction notebook.
'''

import ee
import os
import pandas as pd
import time 
import matplotlib.pyplot as plt
print('Libraries loaded.')


Libraries loaded.


In [2]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [3]:
# Imports and configurations -----------------------------------------------------------------------

YEARS = range(2003, 2026)
SCALE = 11132
ERA5_COLLECTION = 'ECMWF/ERA5_LAND/DAILY_AGGR'
BANDS = ['temperature_2m', 'total_precipitation_sum']

# Output paths and directories -----------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac

save_dir = os.path.join(BASE_OUT_DIR, 'inputs', 'raw_data', 'era5')
print(save_dir)

C:\Users\ibekar\Documents\GitProjects\TGPF\inputs\raw_data\era5


In [4]:
# LOAD MEDITERRANEAN BASIN ECOREGIONS --------------------------------------------------------------
med_bbox = ee.Geometry.BBox(-10, 28, 42, 48)

ecoregions_med = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(med_bbox)

n_eco    = ecoregions_med.size().getInfo()
eco_list = ecoregions_med.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']).getInfo()

print(f'Number of ecoregions intersecting Mediterranean bounding box: {n_eco}')
for f in eco_list['features']:
    p = f['properties']
    print(p['ECO_ID'], '|', p['ECO_NAME'], '|', p['BIOME_NAME'])

# BUILD ECO RECORDS --------------------------------------------------------------------------------
eco_records = []
for f in eco_list['features']:
    p = f['properties']
    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : ee.Geometry(f['geometry'])
    })

print(f'Built {len(eco_records)} ecoregion records.')

# SUBSETTING (set to None to disable) --------------------------------------------------------------
TEST_N   = None
TEST_IDS = None

eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running on {len(eco_run)} / {len(eco_records)} ecoregions.')

Number of ecoregions intersecting Mediterranean bounding box: 59
701 | Mediterranean conifer and mixed forests | Temperate Conifer Forests
822 | East Sahara Desert | Deserts & Xeric Shrublands
833 | North Saharan Xeric Steppe and Woodland | Deserts & Xeric Shrublands
836 | Red Sea coastal desert | Deserts & Xeric Shrublands
845 | West Sahara desert | Deserts & Xeric Shrublands
745 | Saharan halophytics | Flooded Grasslands & Savannas
744 | Nile Delta flooded savanna | Flooded Grasslands & Savannas
758 | Mediterranean High Atlas juniper steppe | Montane Grasslands & Shrublands
648 | Cantabrian mixed forests | Temperate Broadleaf & Mixed Forests
676 | Pyrenees conifer and mixed forests | Temperate Broadleaf & Mixed Forests
788 | Corsican montane broadleaf and mixed forests | Mediterranean Forests, Woodlands & Scrub
789 | Crete Mediterranean forests | Mediterranean Forests, Woodlands & Scrub
792 | Iberian conifer forests | Mediterranean Forests, Woodlands & Scrub
793 | Iberian sclerophyll

In [5]:
def extract_era5_for_ecoregion(eco_feature):
    '''Extract ERA5 data for a single eco-region feature.'''
    eco_id = eco_feature['eco_id']
    eco_name = eco_feature['eco_name']
    geometry = eco_feature['geometry']
    geometry_simplified = geometry.simplify(500)  # ← simplify once here

    # Filter ERA5 collection by date and geometry
    era5 = (ee.ImageCollection(ERA5_COLLECTION)
            .filterDate(f'{YEARS[0]}-01-01', f'{YEARS[-1]}-12-31')
            .select(BANDS))
    
    # Define a per image mapping function

    def image_to_feature(image):
        stats = image.reduceRegion(
            reducer = ee.Reducer.mean(),
            geometry = geometry_simplified, # Simplify geometry to speed up processing
            scale = SCALE, # ERA5-Land native resolution ~0.1°
            maxPixels = 1e9,
            bestEffort = True # Allow processing of large geometries
        )
        return ee.Feature(None, {
            "eco_id" : eco_id,
            "date" : image.date().format('YYYY-MM-dd'),
            "eco_name" : eco_name,
            "temperature_K" : stats.get('temperature_2m'),
            "precip_m" : stats.get('total_precipitation_sum')
        })
    daily_fc = ee.FeatureCollection(era5.map(image_to_feature))
    return daily_fc

In [ ]:
n_submitted = 0
os.makedirs(save_dir, exist_ok=True)


for e in eco_run:
    task_desc = f'{e["eco_id"]} {e["eco_name"]}'
    out_path = os.path.join(save_dir, f'ERA5_eco_{e["eco_id"]}_{e["eco_name"].replace(" ", "_")}.csv')
    print(f'Submitting ERA5 extraction task for {task_desc}...')

    if os.path.exists(out_path):
        print(f'  → Output file already exists. Skipping extraction. ({out_path})')
        continue

    daily_fc = extract_era5_for_ecoregion(e)
    
    records = []
    t0 = time.time()

    for year in YEARS:
        daily_fc_year = daily_fc.filter(ee.Filter.stringContains('date', str(year)))

        try:
            result = daily_fc_year.getInfo() # Trigger computation and results a dict
        except Exception as ex:
            print(f' - Failed for year {year} of {task_desc}: {ex}')
            continue
        features = result['features']
        
        for f in features:
            p = f['properties']
            records.append({
                'eco_id' : p['eco_id'],
                'date' : p['date'],
                'eco_name' : p['eco_name'],
                'temperature_K' : p['temperature_K'],
                'precip_m' : p['precip_m']
            })
        t1 = time.time()
        print(f'  → {year}: {len(features)} days (Time: {t1 - t0:.2f} s)')
    
    df = pd.DataFrame(records)
    df.to_csv(out_path, index=False)
    t2 = time.time()
    print(f'  → Total time for {task_desc}: {t2 - t0:.2f} s')
    print(f'ERA5 extraction task for {task_desc} completed and saved.')

    n_submitted += 1

print(f'All ERA5 extraction tasks completed for {n_submitted} ecoregions.')
print(f'ERA5 data saved to: {save_dir}')



Submitting ERA5 extraction task for 701 Mediterranean conifer and mixed forests...
  → 2003: 365 days (Time: 48.08 s)
  → 2004: 366 days (Time: 95.01 s)
  → 2005: 365 days (Time: 129.61 s)
  → 2006: 365 days (Time: 185.79 s)
  → 2007: 365 days (Time: 229.39 s)
  → 2008: 366 days (Time: 271.11 s)


In [ ]:
PLOT_ECO_ID = 701  # change this to inspect any ecoregion

plot_path = os.path.join(save_dir, f'ERA5_eco_{PLOT_ECO_ID}.csv')
out = pd.read_csv(plot_path)
out["date"] = pd.to_datetime(out["date"])

plt.figure(figsize=(12, 6))
plt.suptitle(f"ERA5 Daily Data for Ecoregion {out['eco_name'].iloc[0]}", fontsize=16)
plt.subplot(2, 1, 1)
plt.plot(out["date"], out["precip_m"])
plt.title("Daily Precipitation (m)")
plt.subplot(2, 1, 2)
plt.plot(out["date"], out["temperature_K"])
plt.title("Daily Temperature (K)")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()
